# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

This dataset contains survey data on mental health indicators among residents of Kilifi County. You will learn to load the metadata, explore the structure, extract data, and perform basic exploratory data analysis (EDA) using programmatic access to all fields and records by their Croissant `@id` values.

### Dataset Source
The dataset source is provided via a [Croissant Schema JSON-LD](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset loaded: {metadata['name']}")
print(f"Description: {metadata['description']}\n")
print(f"Published: {metadata['datePublished']}")
print(f"Version: {metadata['version']}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All dataset structure is referenced by `@id` per the Croissant schema.

In [ ]:
# Access and print record sets, their ids, and the corresponding fields (by @id)
record_sets = [rset for rset in dataset.record_sets()]
print(f"Number of record sets: {len(record_sets)}\n")

for rset in record_sets:
    print(f"RecordSet @id: {rset.id}")
    print(f"  name: {rset.name}")
    print(f"  description: {rset.description}")

    # List all fields in this record set
    print("  Fields and Columns (@id):")
    for field in rset.fields:
        print(f"    Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'n/a')}")

    # If relevant, also print column @ids if present
    if hasattr(rset, 'columns') and rset.columns:
        for col in rset.columns:
            print(f"      Column @id: {col.id}, name: {col.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# --- Choose the correct record_set @id for the main survey ---
# (Replace below with the discovered @id for the main survey responses)

# For demonstration, we will use the first record set
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id

print(f"Using main record set: {main_record_set_id}")

# Load all records as a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {df.shape[0]} records with columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by a numeric field (`@id`), normalizing, and grouping. Use field or column `@id` for any operation.

In [ ]:
# Find a numeric field @id (e.g., PHQ-9 or GAD-7 score)
# Print columns for selection
print('DataFrame columns:', df.columns.tolist())

# For example, suppose the PHQ-9 score column has @id: 'phq9_score' (replace with actual @id if different)
# You can update numeric_field_id below to any other numeric field id as appropriate
numeric_field_id = [col for col in df.columns if 'phq' in col.lower() and 'score' in col.lower()]
numeric_field_id = numeric_field_id[0] if numeric_field_id else df.columns[0]  # fallback to first column

print(f"Selected numeric field for analysis: {numeric_field_id}")

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records (first 5 shown):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field (e.g., 'gender' or 'village') by its @id
group_field_id = [col for col in df.columns if 'gender' in col.lower() or 'village' in col.lower()]
if group_field_id:
    group_field_id = group_field_id[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by '{group_field_id}':")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found in columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, compare groups (e.g., mean scores by gender or village).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of scores
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group if exists
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, you loaded the FAIR² Mental Health Survey Dataset from Kilifi County, explored its record sets, examined fields and columns by their Croissant `@id`, and performed basic exploratory data analysis using the `mlcroissant` library. This approach supports reproducible, schema-driven access to AI-ready data, facilitating wider use, analysis, and responsible AI development.

You are encouraged to further explore and document the variables, relationships, and distributions in this rich dataset for your specific research or application.